# Indirect RI coverage audit — run notebook
Run the cells in order. Change only **PART** in cell 1.

| PART | What it runs | Approx. time (4 CPU cores) |
|---|---|---|
| `A1` | refineR + reflimR, first half of the 11,200 datasets | ~4 h |
| `A2` | refineR + reflimR, second half | ~4 h |
| `B`  | refineR with 50 bootstraps on 140 TSH datasets | ~6–7 h |
| `C`  | RINet (+50 bootstraps) on all datasets | ~1 h |

Each part saves results continuously and can be resumed by re-running the same PART.
At the end, download the zip file shown by the last cell and send it back.

In [ ]:
PART = "A1"          # <-- change to A1, A2, B or C
HOURS_LIMIT = 11.0    # stop safely before the 12 h session limit

import os, glob, subprocess, sys, multiprocessing, shutil, time
CORES = multiprocessing.cpu_count()
hits = glob.glob('/kaggle/input/**/meta.csv', recursive=True) + glob.glob('/content/**/meta.csv', recursive=True) + glob.glob('./**/meta.csv', recursive=True)
assert hits, "meta.csv not found: upload/unzip the bundle first"
DATA = os.path.dirname(hits[0])
OUT = '/kaggle/working/ri_results' if os.path.exists('/kaggle/working') else os.path.abspath('ri_results')
os.makedirs(OUT, exist_ok=True)
print('DATA =', DATA, '| OUT =', OUT, '| CORES =', CORES)
print(sorted(os.listdir(DATA)))

In [ ]:
# Install R and the R packages (only needed for parts A1, A2, B). Takes a few minutes the first time.
def sh(cmd):
    print('$', cmd); p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((p.stdout + p.stderr)[-1500:]); return p.returncode
if PART in ('A1', 'A2', 'B'):
    if shutil.which('Rscript') is None:
        sh('apt-get update -qq && apt-get install -y -qq r-base > /dev/null')
    sh('''Rscript -e 'pk <- c("refineR","reflimR","data.table","future"); miss <- pk[!sapply(pk, requireNamespace, quietly=TRUE)]; if(length(miss)) install.packages(miss, repos="https://cloud.r-project.org", Ncpus=4); for(p in pk) cat(p, as.character(packageVersion(p)), "\\n")' ''')
else:
    print('Part C uses Python only.')

In [ ]:
# Run the selected part (output streams below; safe to re-run to resume)
def stream(cmd):
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(k in line for k in ('oneDNN', 'cpu_feature_guard', 'absl::', 'rebuild TensorFlow')):
            print(line, end='', flush=True)
    return p.wait()
W = os.path.join(DATA, 'worker.R')
if PART == 'A1': rc = stream(['Rscript', W, 'fast', DATA, OUT, '1', '2', str(CORES), str(HOURS_LIMIT)])
elif PART == 'A2': rc = stream(['Rscript', W, 'fast', DATA, OUT, '2', '2', str(CORES), str(HOURS_LIMIT)])
elif PART == 'B': rc = stream(['Rscript', W, 'boot', DATA, OUT, '1', '1', str(CORES), str(HOURS_LIMIT)])
elif PART == 'C': rc = stream([sys.executable, os.path.join(DATA, 'rinet_runner.py'), DATA, os.path.join(DATA, 'rinet_models'), OUT, '50'])
print('exit code', rc)

In [ ]:
# Pack the results to send back
z = shutil.make_archive(os.path.join(os.path.dirname(OUT), f'ri_results_{PART}'), 'zip', OUT)
print('Download this file and send it:', z)
for f in sorted(os.listdir(OUT)): print(f, os.path.getsize(os.path.join(OUT, f)), 'bytes')